# Piper Clean 100 试听测试

在 Colab 或本地 Jupyter 中从上到下运行。本地用 uv 管理的虚拟环境（默认不带 pip）时，安装单元会自动改用 `uv pip install`。优先用 GPU 合成（自动检测 TensorRT/CUDA/CoreML/DirectML），无 GPU 时回退 CPU，共 **18 条 WAV**：新唤醒词 `hi circle` 和旧唤醒词 `hey hi` 各 9 条，覆盖三个说话人 ID 和三档语速。

`hi circle` 的目标读音是 **hi CIR-cle**，重音在 `circle` 首音节。音标 `hˈaɪ sˈɜːkəl` 与模型自带 espeak 音素化（`voice.phonemize('hi circle')`）的输出一致。`hey hi` 保留作对照基准，同一说话人和语速下可直接比较。

作者发布页写有 244 位说话人，随模型发布的配置列有 245 个 ID。这里试听 ID 0、122 和 244，验证末尾 ID 也能生成音频。音频保存到当前工作目录的 `piper_clean100_test/wake_word_preview/`。

[Clean 100 权重与作者声明](https://brycebeattie.com/files/tts/) · [LibriTTS-R 数据来源](https://www.openslr.org/141/) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [1]:
import importlib.util
import shutil
import subprocess
import sys

if importlib.util.find_spec("piper") is None:
    has_pip = (
        shutil.which("uv") is None
        or subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode == 0
    )
    if has_pip:
        subprocess.run([sys.executable, "-m", "pip", "install", "piper-tts==1.3.0"], check=True)
    else:
        # uv 创建的虚拟环境默认不带 pip，改用 uv 安装
        subprocess.run(["uv", "pip", "install", "--python", sys.executable, "piper-tts==1.3.0"], check=True)
else:
    print("piper-tts 已安装，跳过。")

piper-tts 已安装，跳过。


In [2]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

from piper import PiperVoice

work_dir = Path.cwd() / "piper_clean100_test"
work_dir.mkdir(exist_ok=True)
source_url = "https://sfo3.digitaloceanspaces.com/bkmdls"
model_hashes = {
    "clean100.onnx": "a48b44084f5465a9d1c5e74c46c6ce6e1907af85e34f15292b9f44b13263f6e4",
    "clean100.onnx.json": "575a90e0f03154d30c4e8d9b1a86dcb590f7df2fc65e24466dde3c91491984b8",
}
for filename, expected_hash in model_hashes.items():
    path = work_dir / filename
    if path.is_file():
        if sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise ValueError(f"{path} 的 SHA-256 不匹配，请先移走该文件。")
    else:
        with TemporaryDirectory(dir=work_dir) as staging:
            downloaded = Path(staging) / filename
            urlretrieve(f"{source_url}/{filename}", downloaded)
            if sha256(downloaded.read_bytes()).hexdigest() != expected_hash:
                raise ValueError(f"{filename} 下载后的 SHA-256 不匹配。")
            downloaded.rename(path)

# 与 microwakeword/audio/piper_clean100.py 的 load_clean100 一致：优先 GPU，回退 CPU。
import json

import onnxruntime
from piper import PiperConfig, PiperVoice

GPU_PROVIDERS = (
    "TensorrtExecutionProvider",
    "CUDAExecutionProvider",
    "CoreMLExecutionProvider",
    "DmlExecutionProvider",
)
available = set(onnxruntime.get_available_providers())
providers = [name for name in GPU_PROVIDERS if name in available] + ["CPUExecutionProvider"]
model_path = work_dir / "clean100.onnx"
with open(f"{model_path}.json", "r", encoding="utf-8") as config_file:
    config = PiperConfig.from_dict(json.load(config_file))
session = onnxruntime.InferenceSession(
    str(model_path),
    sess_options=onnxruntime.SessionOptions(),
    providers=providers,
)
print(f"Clean 100 · 推理设备：{session.get_providers()[0]}")
voice = PiperVoice(config=config, session=session)
print(f"ONNX 配置声明 {voice.config.num_speakers} 个说话人 ID，采样率 {voice.config.sample_rate} Hz。")

In [9]:
from statistics import median
from unicodedata import normalize
import wave

import numpy as np
from piper import SynthesisConfig

phrases = {
    "hi_circle": "hˈaɪ sˈɜːkəl"
}
length_scales = {"fast": 0.65, "normal": 1.0, "slow": 1.25}
speaker_ids = (0, 122, 244)
wav_dir = work_dir / "wake_word_preview"
wav_dir.mkdir(exist_ok=True)
if not set(speaker_ids) <= set(voice.config.speaker_id_map.values()):
    raise ValueError(f"存在模型配置未列出的说话人 ID。请检查 {speaker_ids}。")

wav_paths = {}
durations = {}
pcm_hashes = set()
for phrase, ipa in phrases.items():
    phonemes = list(normalize("NFD", ipa))
    missing = set(phonemes) - voice.config.phoneme_id_map.keys()
    if missing:
        raise ValueError(f"{phrase} 包含模型未定义的音素 {sorted(missing)}。")
    phoneme_ids = voice.phonemes_to_ids(phonemes)
    for speed, length_scale in length_scales.items():
        for speaker_id in speaker_ids:
            config = SynthesisConfig(
                speaker_id=speaker_id, length_scale=length_scale,
                noise_scale=0.667, noise_w_scale=0.8,
            )
            audio = voice.phoneme_ids_to_audio(phoneme_ids, config)
            peak = np.max(np.abs(audio), initial=0)
            if not np.isfinite(peak) or peak == 0:
                raise ValueError(f"{phrase} / {speed} / ID {speaker_id} 输出无效。")
            pcm = np.rint(audio / peak * 32767).astype("<i2").tobytes()
            path = wav_dir / f"{phrase}_{speed}_speaker_{speaker_id:03d}.wav"
            with wave.open(str(path), "wb") as wav:
                wav.setparams((1, 2, voice.config.sample_rate, 0, "NONE", ""))
                wav.writeframes(pcm)

            key = (phrase, speed, speaker_id)
            wav_paths[key] = path
            durations[key] = len(audio) / voice.config.sample_rate
            pcm_hashes.add(sha256(pcm).hexdigest())

expected = len(phrases) * len(length_scales) * len(speaker_ids)
assert len(wav_paths) == len(pcm_hashes) == expected, "试听样本缺失或 PCM 重复"
print(f"已生成 {expected} 条不同的试听 WAV，保存在 {wav_dir}。")
for phrase in phrases:
    for speed in length_scales:
        seconds = median(durations[phrase, speed, sid] for sid in speaker_ids)
        print(f"{phrase:10} {speed:6} 时长中位数 {seconds:.2f} 秒")

已生成 9 条不同的试听 WAV，保存在 /Users/program/micro-wake-word/notebooks/piper_clean100_test/wake_word_preview。
hi_circle  fast   时长中位数 0.65 秒
hi_circle  normal 时长中位数 0.86 秒
hi_circle  slow   时长中位数 1.01 秒


In [12]:
from IPython.display import Audio, display

listen_speaker = 244  # 也可选 122 或 244
for speed, scale in length_scales.items():
    for phrase in phrases:
        key = (phrase, speed, listen_speaker)
        print(f"{phrase} · {speed} · length_scale={scale} · {durations[key]:.2f} 秒")
        display(Audio(filename=str(wav_paths[key]), normalize=False))

hi_circle · fast · length_scale=0.65 · 0.60 秒


hi_circle · normal · length_scale=1.0 · 0.79 秒


hi_circle · slow · length_scale=1.25 · 1.01 秒
